<a href="https://colab.research.google.com/github/okothjosh/Machine_learning_projects/blob/main/01_data_ingestion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [24]:
!pip install gtfs-kit geopandas prophet xgboost lightgbm tensorflow streamlit folium pydeck

In [25]:
import os
import zipfile

# Ensure project_base_dir is defined from previous steps
project_base_dir = '/content/drive/MyDrive/matatu_ml/'

# Define raw and processed data directories
raw_data_dir = os.path.join(project_base_dir, 'data', 'raw')
processed_data_dir = os.path.join(project_base_dir, 'data', 'processed')

# Create directories if they don't exist
os.makedirs(raw_data_dir, exist_ok=True)
os.makedirs(processed_data_dir, exist_ok=True)

# Use the GTFS zip file provided by the user
gtfs_zip_path = '/content/gtfs.zip'

# Unzip the GTFS data
# Check if any GTFS feed (identified by routes.txt) exists in raw_data_dir already
found_feed_dir_after_check = None
for root, dirs, files in os.walk(raw_data_dir):
    if 'routes.txt' in files:
        found_feed_dir_after_check = root
        break

if found_feed_dir_after_check is None: # Only unzip if not already extracted
    print(f"Extracting GTFS data from {gtfs_zip_path} to {raw_data_dir}...")
    try:
        with zipfile.ZipFile(gtfs_zip_path, 'r') as zip_ref:
            zip_ref.extractall(raw_data_dir) # Extract all contents into raw_data_dir
        print("Extraction complete.")
    except zipfile.BadZipFile:
        print(f"Error: {gtfs_zip_path} is not a valid zip file. Please check the file.")
        raise # Critical error, cannot proceed
    except FileNotFoundError:
        print(f"Error: Zip file not found at {gtfs_zip_path}. Please ensure it is in /content/.")
        raise # Critical error, cannot proceed
else:
    print(f"GTFS data appears to be already extracted in {found_feed_dir_after_check}.")

# Identify the actual GTFS feed directory (it might be nested one level deeper or named differently)
actual_gtfs_feed_dir = None
# Look for a directory containing 'routes.txt' within raw_data_dir after extraction
for root, dirs, files in os.walk(raw_data_dir):
    if 'routes.txt' in files:
        actual_gtfs_feed_dir = root
        break

if actual_gtfs_feed_dir is None:
    print("Error: Could not find 'routes.txt' in any subdirectory of raw_data_dir after extraction.")
    print("Please verify the contents of the extracted GTFS data.")
    # Fallback or raise error
    raise FileNotFoundError("GTFS feed directory not found.")

print(f"Actual GTFS feed directory identified: {actual_gtfs_feed_dir}")

GTFS data appears to be already extracted in /content/drive/MyDrive/matatu_ml/data/raw.
Actual GTFS feed directory identified: /content/drive/MyDrive/matatu_ml/data/raw


### 2. Parse GTFS Data with `gtfs-kit` and Export to CSV

Now, I will use `gtfs-kit` to read the GTFS feed and then export the `routes`, `stops`, `stop_times`, and `shapes` tables into individual CSV files. These CSVs will be saved in the `data/processed` folder.

In [26]:
import gtfs_kit as gk
import pandas as pd

# Load the GTFS feed
print(f"Loading GTFS feed from {actual_gtfs_feed_dir}...")
feed = gk.read_feed(actual_gtfs_feed_dir, dist_units='mi')
print("GTFS feed loaded successfully.")

# Define the tables to export
tables_to_export = {
    'routes': feed.routes,
    'stops': feed.stops,
    'stop_times': feed.stop_times,
    'shapes': feed.shapes
}

# Export each table to a CSV file
for table_name, df in tables_to_export.items():
    output_path = os.path.join(processed_data_dir, f'{table_name}.csv')
    print(f"Exporting {table_name} to {output_path}...")
    df.to_csv(output_path, index=False)
    print(f"Exported {len(df)} rows to {output_path}")

print("All specified GTFS tables exported to CSV successfully.")

Loading GTFS feed from /content/drive/MyDrive/matatu_ml/data/raw...
GTFS feed loaded successfully.
Exporting routes to /content/drive/MyDrive/matatu_ml/data/processed/routes.csv...
Exported 136 rows to /content/drive/MyDrive/matatu_ml/data/processed/routes.csv
Exporting stops to /content/drive/MyDrive/matatu_ml/data/processed/stops.csv...
Exported 4273 rows to /content/drive/MyDrive/matatu_ml/data/processed/stops.csv
Exporting stop_times to /content/drive/MyDrive/matatu_ml/data/processed/stop_times.csv...
Exported 7533 rows to /content/drive/MyDrive/matatu_ml/data/processed/stop_times.csv
Exporting shapes to /content/drive/MyDrive/matatu_ml/data/processed/shapes.csv...
Exported 36483 rows to /content/drive/MyDrive/matatu_ml/data/processed/shapes.csv
All specified GTFS tables exported to CSV successfully.


In [27]:
import json
import pandas as pd

# Path to the provided JSON file
json_file_path = '/content/nairobi_weather_2024.json'

print(f"Loading weather data from {json_file_path}...")

# Load the JSON data from the file
with open(json_file_path, 'r') as f:
    weather_data = json.load(f)

# Extract hourly data
hourly_df = pd.DataFrame(weather_data['hourly'])

# Convert time to datetime objects
hourly_df['time'] = pd.to_datetime(hourly_df['time'])

# Rename columns for clarity (assuming similar structure to Open-Meteo API)
hourly_df.rename(columns={
    'precipitation': 'rainfall_mm',
    'temperature_2m': 'temperature_c',
    'wind_speed_10m': 'wind_speed'
}, inplace=True)

print("Weather data loaded and processed successfully from JSON file.")

# Display the first 5 rows of the DataFrame
display(hourly_df.head())

Loading weather data from /content/nairobi_weather_2024.json...
Weather data loaded and processed successfully from JSON file.


,time,temperature_c,rain,wind_speed
0,2024-01-01 00:00:00,15.4,0.0,4.6
1,2024-01-01 01:00:00,16.6,0.0,3.6
2,2024-01-01 02:00:00,15.8,0.0,6.2
3,2024-01-01 03:00:00,15.4,0.0,6.4
4,2024-01-01 04:00:00,16.4,0.0,4.4


In [28]:
import geopandas
from shapely.geometry import LineString, Point
import osmnx as ox

# Load the shapes data which defines the route geometries
shapes_df = pd.read_csv(os.path.join(processed_data_dir, 'shapes.csv'))

print("Shapes data loaded successfully.")

# Group by shape_id and create LineString geometries for each shape
def create_linestring(group):
    return LineString(group[['shape_pt_lon', 'shape_pt_lat']].values)

# Sort by sequence to ensure correct line order
shapes_df = shapes_df.sort_values(by=['shape_id', 'shape_pt_sequence'])

# Create a GeoDataFrame from the LineStrings
# Use 'EPSG:4326' (WGS 84) as the CRS, which is standard for lat/lon
# We'll create a pandas DataFrame first and then convert to GeoDataFrame explicitly setting the geometry
geometries = shapes_df.groupby('shape_id').apply(create_linestring)
routes_gdf = pd.DataFrame({'geometry': geometries}).reset_index()
routes_gdf = geopandas.GeoDataFrame(routes_gdf, geometry='geometry', crs="EPSG:4326")

print("Route geometries created and stored in GeoDataFrame.")
display(routes_gdf.head())

Shapes data loaded successfully.
Route geometries created and stored in GeoDataFrame.


/tmp/ipykernel_1210/372260388.py:20: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  geometries = shapes_df.groupby('shape_id').apply(create_linestring)


,shape_id,geometry
0,10106110,"LINESTRING (36.75878 -1.17384, 36.75902 -1.174..."
1,10106111,"LINESTRING (36.8225 -1.28142, 36.82238 -1.2813..."
2,10107110,"LINESTRING (36.73198 -1.19437, 36.73557 -1.194..."
3,10107111,"LINESTRING (36.82478 -1.28304, 36.82583 -1.281..."
4,10108110,"LINESTRING (36.76672 -1.21956, 36.76841 -1.220..."


In [54]:
import osmnx as ox
import geopandas as gpd
from shapely.geometry import LineString, Point, Polygon, MultiPolygon
import pandas as pd
import uuid
import os

# Define cache directory
cache_dir = os.path.join(processed_data_dir, 'cache')
os.makedirs(cache_dir, exist_ok=True)

# Path for the combined cached POIs GeoDataFrame
combined_pois_cache_path = os.path.join(cache_dir, 'all_pois_gdf.parquet')

# List of columns expected in the final all_pois_gdf
# This helps in creating an empty GeoDataFrame if no POIs are found
poi_cols_to_keep = ['osm_id', 'geometry', 'element_type', 'name', 'amenity', 'shop', 'office', 'building', 'healthcare', 'shape_id']

# Try to load from combined cache first
if os.path.exists(combined_pois_cache_path):
    print(f"Loading all POIs from cache: {combined_pois_cache_path}")
    all_pois_gdf = gpd.read_parquet(combined_pois_cache_path)
    print(f"Loaded {len(all_pois_gdf)} POIs from cache.")
else:
    print("Combined POIs cache not found. Proceeding with individual POI extraction and caching...")

    # Project the GeoDataFrame to a local CRS to perform accurate buffering in meters
    print("Projecting routes_gdf to a local UTM CRS (EPSG:32737) for accurate buffering...")
    routes_gdf_proj = routes_gdf.to_crs(epsg=32737)

    # Create a 500-meter buffer around each route
    buffer_distance_m = 500
    print(f"Creating {buffer_distance_m} meter buffer around each route...")
    routes_buffer = routes_gdf_proj.buffer(buffer_distance_m)

    # Convert the buffered geometries back to EPSG:4326 for Overpass API query
    routes_buffer_gdf = gpd.GeoDataFrame(geometry=routes_buffer, crs=32737).to_crs(epsg=4326)

    print("Buffers created and reprojected. Querying OSM for POIs...")

    # Define the POI tags to query
    poi_tags = {
        'shop': True,
        'amenity': ['school', 'hospital', 'pharmacy', 'university', 'college'],
        'office': True,
        'building': ['hospital', 'school', 'university', 'college', 'office', 'government', 'commercial'],
        'healthcare': ['hospital', 'clinic', 'pharmacy']
    }

    # Initialize a list to store GeoDataFrames of POIs for concatenation
    all_pois_list = []

    # Iterate through each route's buffer and query POIs
    for index, row in routes_buffer_gdf.iterrows():
        route_shape_id = routes_gdf.loc[index, 'shape_id']
        buffer_polygon = row.geometry

        # Define cache path for this specific route's POIs
        single_route_cache_path = os.path.join(cache_dir, f'pois_shape_{route_shape_id}.parquet')

        if os.path.exists(single_route_cache_path):
            print(f"Loading POIs for shape_id {route_shape_id} from cache: {single_route_cache_path}")
            current_pois = gpd.read_parquet(single_route_cache_path)
            all_pois_list.append(current_pois)
            continue # Move to next route if loaded from cache

        # If not in cache, proceed with OSMnx query
        if buffer_polygon.is_empty:
            continue

        try:
            # Ensure the polygon is valid for osmnx
            if not buffer_polygon.is_valid:
                buffer_polygon = buffer_polygon.buffer(0) # Attempt to fix invalid polygons

            pois = ox.features_from_polygon(buffer_polygon, tags=poi_tags)

            if not pois.empty:
                # OSMnx often returns osm_id as index. Reset index to make it a column.
                if pois.index.name == 'osm_id':
                    pois = pois.reset_index()

                # Ensure 'osm_id' column exists. If not, create it and fill with pd.NA
                if 'osm_id' not in pois.columns:
                    pois['osm_id'] = pd.NA

                pois['shape_id'] = route_shape_id

                # Fill any remaining NaN osm_ids with a unique placeholder string.
                # This prevents dropping these rows due to missing OSM IDs in cleaning.
                null_osm_mask = pois['osm_id'].isnull()
                if null_osm_mask.any():
                    # Generate unique IDs for nulls using UUID for global uniqueness
                    pois.loc[null_osm_mask, 'osm_id'] = [f"MISSING_OSMID_{route_shape_id}_{i}_{uuid.uuid4().hex[:8]}"
                                                         for i in range(null_osm_mask.sum())]

                # Ensure all required columns are present, add if missing and fill with pd.NA
                for col in poi_cols_to_keep:
                    if col not in pois.columns:
                        pois[col] = pd.NA

                current_pois = pois[poi_cols_to_keep] # Select only the desired and ensured columns

                # Cache this individual route's POIs
                current_pois.to_parquet(single_route_cache_path, index=False)
                print(f"Cached POIs for shape_id {route_shape_id} to {single_route_cache_path}")

                all_pois_list.append(current_pois)

        except Exception as e:
            print(f"Error querying POIs for shape_id {route_shape_id}: {e}")

    # Concatenate all GeoDataFrames from the list
    if all_pois_list:
        all_pois_gdf = pd.concat(all_pois_list, ignore_index=True)
        # Save the combined POIs GeoDataFrame to a master cache file
        all_pois_gdf.to_parquet(combined_pois_cache_path, index=False)
        print(f"Saved combined POIs to master cache: {combined_pois_cache_path}")
    else:
        # Create an empty GeoDataFrame with correct columns and CRS if no POIs were found
        all_pois_gdf = gpd.GeoDataFrame(columns=poi_cols_to_keep, geometry='geometry', crs="EPSG:4326")
        print("No POIs found for any routes after processing.")

print(f"Found {len(all_pois_gdf)} POIs across all routes.")
display(all_pois_gdf.head())

# Save the POIs to a CSV file (as originally requested)
output_pois_path = os.path.join(processed_data_dir, 'routes_pois.csv')
all_pois_gdf.to_csv(output_pois_path, index=False)
print(f"All POIs saved to {output_pois_path}")

Loading all POIs from cache: /content/drive/MyDrive/matatu_ml/data/processed/cache/all_pois_gdf.parquet
Loaded 91123 POIs from cache.
Found 91123 POIs across all routes.


,osm_id,geometry,element_type,name,amenity,shop,office,building,healthcare,shape_id
0,MISSING_OSMID_10106110_0_5ee2a577,POINT (36.81492 -1.27217),None,Aga Khan Nursery,school,None,None,None,None,10106110
1,MISSING_OSMID_10106110_1_7423d94a,POINT (36.80451 -1.23001),None,City Walk - Village Market,None,shoes,None,None,None,10106110
2,MISSING_OSMID_10106110_2_bad06261,POINT (36.80146 -1.22415),None,River Garden Centre,None,garden_centre,None,None,None,10106110
3,MISSING_OSMID_10106110_3_ba58e30a,POINT (36.82025 -1.24999),None,Onn The Way Supermarket Muthaiga,None,supermarket,None,None,None,10106110
4,MISSING_OSMID_10106110_4_03c59614,POINT (36.80496 -1.22869),None,Dorman,None,mall,None,None,None,10106110


All POIs saved to /content/drive/MyDrive/matatu_ml/data/processed/routes_pois.csv


### 4. Load EPRA Monthly Diesel/Petrol Prices

Now, I will load the monthly diesel and petrol prices from the provided CSV file into a pandas DataFrame.

In [31]:
epra_prices_path = '/content/Pump Prices  Energy and Petroleum Regulatory Authority.csv'
print(f"Loading EPRA prices from {epra_prices_path}...")
epra_df = pd.read_csv(epra_prices_path)
print("EPRA prices loaded successfully.")
display(epra_df.head())

Loading EPRA prices from /content/Pump Prices  Energy and Petroleum Regulatory Authority.csv...
EPRA prices loaded successfully.


,From,To,Town,Super (PMS),Diesel (AGO)
0,15-12-2025,14-01-2026,Nairobi,184.52,171.47
1,15-11-2025,14-12-2025,Nairobi,184.52,171.47
2,15-10-2025,14-11-2025,Nairobi,184.52,171.47
3,15-09-2025,14-10-2025,Nairobi,184.52,171.47
4,15-08-2025,14-09-2025,Nairobi,185.31,171.58


### 5. Schema Validation

#### Validate EPRA Monthly Diesel/Petrol Prices (`epra_df`)

In [32]:
print('--- Validating epra_df ---')

# 1. Assert row counts (example: expecting at least 12 months data for 2 years = 24 rows)
expected_min_rows_epra = 12
if len(epra_df) < expected_min_rows_epra:
    print(f"Warning: epra_df has {len(epra_df)} rows, expected at least {expected_min_rows_epra}.")
else:
    print(f"epra_df row count ({len(epra_df)}) is acceptable.")

# 2. Check for expected columns
expected_epra_columns = ['From', 'To', 'Town', 'Super (PMS)', 'Diesel (AGO)']
missing_epra_cols = [col for col in expected_epra_columns if col not in epra_df.columns]
if missing_epra_cols:
    raise ValueError(f"Missing expected columns in epra_df: {missing_epra_cols}")
else:
    print("All expected columns are present in epra_df.")

# 3. Convert 'From' and 'To' columns to datetime and validate date range
epra_df['From_Date'] = pd.to_datetime(epra_df['From'], format='%d-%m-%Y')
epra_df['To_Date'] = pd.to_datetime(epra_df['To'], format='%d-%m-%Y')

# Assuming prices are for Nairobi, 'Town' should ideally be consistent
if not (epra_df['Town'] == 'Nairobi').all():
    print("Warning: 'Town' column in epra_df contains values other than 'Nairobi'.")

print(f"EPRA price data date range: {epra_df['From_Date'].min().strftime('%Y-%m-%d')} to {epra_df['To_Date'].max().strftime('%Y-%m-%d')}")

# 4. Check for nulls in primary key candidates (e.g., 'From_Date' and 'Town')
if epra_df[['From_Date', 'Town']].isnull().any().any():
    raise ValueError("Null values found in primary key candidates ('From_Date', 'Town') in epra_df.")
else:
    print("No nulls found in 'From_Date' or 'Town' in epra_df.")

print('epra_df validation complete.')

--- Validating epra_df ---
epra_df row count (19) is acceptable.
All expected columns are present in epra_df.
EPRA price data date range: 2024-11-15 to 2026-05-14
No nulls found in 'From_Date' or 'Town' in epra_df.
epra_df validation complete.


#### Validate Hourly Weather Data (`hourly_df`)

In [33]:
print('\n--- Validating hourly_df ---')

# 1. Assert row counts (expecting a full year of hourly data, 365*24 = 8760 rows, for 2 years = ~17520)
expected_min_rows_hourly = 8760 * 2 # Minimum for two years
if len(hourly_df) < expected_min_rows_hourly:
    print(f"Warning: hourly_df has {len(hourly_df)} rows, expected at least {expected_min_rows_hourly}. Data might be incomplete.")
else:
    print(f"hourly_df row count ({len(hourly_df)}) is acceptable.")

# 2. Check for expected columns
expected_hourly_columns = ['time', 'temperature_c', 'rain', 'wind_speed'] # 'precipitation' was renamed to 'rainfall_mm'
missing_hourly_cols = [col for col in expected_hourly_columns if col not in hourly_df.columns]
if missing_hourly_cols:
    raise ValueError(f"Missing expected columns in hourly_df: {missing_hourly_cols}")
else:
    print("All expected columns are present in hourly_df.")

# 3. Validate 'time' column as datetime and check its range
if not pd.api.types.is_datetime64_any_dtype(hourly_df['time']):
    raise TypeError("'time' column in hourly_df is not of datetime type.")

print(f"Weather data date range: {hourly_df['time'].min().strftime('%Y-%m-%d %H:%M')} to {hourly_df['time'].max().strftime('%Y-%m-%d %H:%M')}")

# 4. Check for nulls in 'time' (primary key)
if hourly_df['time'].isnull().any():
    raise ValueError("Null values found in 'time' column (primary key) in hourly_df.")
else:
    print("No nulls found in 'time' in hourly_df.")

print('hourly_df validation complete.')


--- Validating hourly_df ---
hourly_df row count (18984) is acceptable.
All expected columns are present in hourly_df.
Weather data date range: 2024-01-01 00:00 to 2026-03-01 23:00
No nulls found in 'time' in hourly_df.
hourly_df validation complete.


#### Validate GTFS `routes` data (`feed.routes`)

In [34]:
print('\n--- Validating feed.routes ---')
routes_df = feed.routes

# 1. Assert row counts (e.g., expecting more than 0 routes)
if len(routes_df) == 0:
    raise ValueError("routes_df has 0 rows. GTFS routes data is empty.")
else:
    print(f"routes_df row count ({len(routes_df)}) is acceptable.")

# 2. Check for expected columns
expected_route_columns = ['route_id', 'agency_id', 'route_short_name', 'route_type']
missing_route_cols = [col for col in expected_route_columns if col not in routes_df.columns]
if missing_route_cols:
    raise ValueError(f"Missing expected columns in routes_df: {missing_route_cols}")
else:
    print("All expected columns are present in routes_df.")

# 3. Check for nulls in primary key ('route_id')
if routes_df['route_id'].isnull().any():
    raise ValueError("Null values found in 'route_id' in routes_df.")
else:
    print("No nulls found in 'route_id' in routes_df.")

print('feed.routes validation complete.')


--- Validating feed.routes ---
routes_df row count (136) is acceptable.
All expected columns are present in routes_df.
No nulls found in 'route_id' in routes_df.
feed.routes validation complete.


#### Validate GTFS `stops` data (`feed.stops`)

In [35]:
print('\n--- Validating feed.stops ---')
stops_df = feed.stops

# 1. Assert row counts (e.g., expecting more than 0 stops)
if len(stops_df) == 0:
    raise ValueError("stops_df has 0 rows. GTFS stops data is empty.")
else:
    print(f"stops_df row count ({len(stops_df)}) is acceptable.")

# 2. Check for expected columns
expected_stop_columns = ['stop_id', 'stop_name', 'stop_lat', 'stop_lon']
missing_stop_cols = [col for col in expected_stop_columns if col not in stops_df.columns]
if missing_stop_cols:
    raise ValueError(f"Missing expected columns in stops_df: {missing_stop_cols}")
else:
    print("All expected columns are present in stops_df.")

# 3. Check for nulls in primary key ('stop_id') and coordinates
if stops_df['stop_id'].isnull().any():
    raise ValueError("Null values found in 'stop_id' in stops_df.")
if stops_df[['stop_lat', 'stop_lon']].isnull().any().any():
    raise ValueError("Null values found in 'stop_lat' or 'stop_lon' in stops_df.")
else:
    print("No nulls found in 'stop_id', 'stop_lat', or 'stop_lon' in stops_df.")

print('feed.stops validation complete.')


--- Validating feed.stops ---
stops_df row count (4273) is acceptable.
All expected columns are present in stops_df.
No nulls found in 'stop_id', 'stop_lat', or 'stop_lon' in stops_df.
feed.stops validation complete.


#### Validate GTFS `stop_times` data (`feed.stop_times`)

In [36]:
print('\n--- Validating feed.stop_times ---')
stop_times_df = feed.stop_times

# 1. Assert row counts (e.g., expecting more than 0 stop times)
if len(stop_times_df) == 0:
    raise ValueError("stop_times_df has 0 rows. GTFS stop_times data is empty.")
else:
    print(f"stop_times_df row count ({len(stop_times_df)}) is acceptable.")

# 2. Check for expected columns
expected_stop_time_columns = ['trip_id', 'arrival_time', 'departure_time', 'stop_id', 'stop_sequence']
missing_stop_time_cols = [col for col in expected_stop_time_columns if col not in stop_times_df.columns]
if missing_stop_time_cols:
    raise ValueError(f"Missing expected columns in stop_times_df: {missing_stop_time_cols}")
else:
    print("All expected columns are present in stop_times_df.")

# 3. Check for nulls in essential columns
if stop_times_df[['trip_id', 'stop_id', 'stop_sequence']].isnull().any().any():
    raise ValueError("Null values found in 'trip_id', 'stop_id', or 'stop_sequence' in stop_times_df.")
else:
    print("No nulls found in 'trip_id', 'stop_id', or 'stop_sequence' in stop_times_df.")

# 4. Validate time format (can be tricky with GTFS, but basic check)
# Assuming times are strings like 'HH:MM:SS'
# A more robust check would convert to timedelta, but for schema, we just check non-null
if stop_times_df['arrival_time'].isnull().any() or stop_times_df['departure_time'].isnull().any():
    print("Warning: Null values found in 'arrival_time' or 'departure_time' in stop_times_df. This might be acceptable depending on GTFS spec.")

print('feed.stop_times validation complete.')


--- Validating feed.stop_times ---
stop_times_df row count (7533) is acceptable.
All expected columns are present in stop_times_df.
No nulls found in 'trip_id', 'stop_id', or 'stop_sequence' in stop_times_df.
feed.stop_times validation complete.


#### Validate GTFS `shapes` data (`feed.shapes`)

In [37]:
print('\n--- Validating feed.shapes ---')
shapes_from_feed_df = feed.shapes # Use a distinct name to avoid confusion with the loaded shapes_df

# 1. Assert row counts (e.g., expecting more than 0 shape points)
if len(shapes_from_feed_df) == 0:
    raise ValueError("shapes_df has 0 rows. GTFS shapes data is empty.")
else:
    print(f"shapes_df row count ({len(shapes_from_feed_df)}) is acceptable.")

# 2. Check for expected columns
expected_shape_columns = ['shape_id', 'shape_pt_lat', 'shape_pt_lon', 'shape_pt_sequence']
missing_shape_cols = [col for col in expected_shape_columns if col not in shapes_from_feed_df.columns]
if missing_shape_cols:
    raise ValueError(f"Missing expected columns in shapes_df: {missing_shape_cols}")
else:
    print("All expected columns are present in shapes_df.")

# 3. Check for nulls in primary key candidates and coordinates
if shapes_from_feed_df[['shape_id', 'shape_pt_lat', 'shape_pt_lon', 'shape_pt_sequence']].isnull().any().any():
    raise ValueError("Null values found in essential columns in shapes_df.")
else:
    print("No nulls found in 'shape_id', 'shape_pt_lat', 'shape_pt_lon', or 'shape_pt_sequence' in shapes_df.")

print('feed.shapes validation complete.')


--- Validating feed.shapes ---
shapes_df row count (36483) is acceptable.
All expected columns are present in shapes_df.
No nulls found in 'shape_id', 'shape_pt_lat', 'shape_pt_lon', or 'shape_pt_sequence' in shapes_df.
feed.shapes validation complete.


#### Validate `routes_gdf` (GeoDataFrame of Route Geometries)

In [38]:
print('\n--- Validating routes_gdf ---')

# 1. Assert row counts (should correspond to number of unique shapes)
num_unique_shapes = shapes_df['shape_id'].nunique()
if len(routes_gdf) != num_unique_shapes:
    print(f"Warning: routes_gdf has {len(routes_gdf)} rows, but there are {num_unique_shapes} unique shape_ids. This might indicate an issue during GeoDataFrame creation.")
else:
    print(f"routes_gdf row count ({len(routes_gdf)}) matches unique shape_ids.")

# 2. Check for expected columns and geometry type
if 'geometry' not in routes_gdf.columns:
    raise ValueError("'geometry' column not found in routes_gdf.")
if not all(isinstance(geom, LineString) for geom in routes_gdf.geometry if geom is not None):
    print("Warning: Not all geometries in routes_gdf are LineString objects.")

# 3. Validate CRS
if routes_gdf.crs != 'EPSG:4326':
    raise ValueError(f"CRS of routes_gdf is {routes_gdf.crs}, expected EPSG:4326.")
else:
    print("routes_gdf has correct CRS (EPSG:4326).")

# 4. Check for null geometries
if routes_gdf['geometry'].isnull().any():
    print("Warning: Null geometries found in routes_gdf.")
else:
    print("No null geometries found in routes_gdf.")

print('routes_gdf validation complete.')


--- Validating routes_gdf ---
routes_gdf row count (272) matches unique shape_ids.
routes_gdf has correct CRS (EPSG:4326).
No null geometries found in routes_gdf.
routes_gdf validation complete.


#### Validate `all_pois_gdf` (GeoDataFrame of POIs)

In [39]:
print('\n--- Validating all_pois_gdf ---')

# 1. Assert row counts (expecting more than 0 POIs if query was successful)
if len(all_pois_gdf) == 0:
    print("Warning: all_pois_gdf has 0 rows. No POIs were found or extracted.")
else:
    print(f"all_pois_gdf row count ({len(all_pois_gdf)}) is acceptable.")

# 2. Check for expected columns and geometry type
expected_poi_gdf_columns = ['osm_id', 'geometry', 'element_type', 'name', 'shape_id']
missing_poi_cols = [col for col in expected_poi_gdf_columns if col not in all_pois_gdf.columns]
if missing_poi_cols:
    print(f"Warning: Missing expected columns in all_pois_gdf: {missing_poi_cols}")

if 'geometry' not in all_pois_gdf.columns:
    raise ValueError("'geometry' column not found in all_pois_gdf.")
# Check if geometries are Points or Polygons (OSMnx returns both)
if not all(geom.geom_type in ['Point', 'Polygon', 'MultiPolygon'] for geom in all_pois_gdf.geometry if geom is not None):
    print("Warning: Not all geometries in all_pois_gdf are Point/Polygon/MultiPolygon objects.")

# 3. Validate CRS
if all_pois_gdf.crs != 'EPSG:4326':
    raise ValueError(f"CRS of all_pois_gdf is {all_pois_gdf.crs}, expected EPSG:4326.")
else:
    print("all_pois_gdf has correct CRS (EPSG:4326).")

# 4. Check for nulls in essential columns
if all_pois_gdf[['osm_id', 'shape_id']].isnull().any().any():
    print("Warning: Null values found in 'osm_id' or 'shape_id' in all_pois_gdf.")
else:
    print("No nulls found in 'osm_id' or 'shape_id' in all_pois_gdf.")

print('all_pois_gdf validation complete.')


--- Validating all_pois_gdf ---
all_pois_gdf row count (91123) is acceptable.
all_pois_gdf has correct CRS (EPSG:4326).
all_pois_gdf validation complete.


In [53]:
print('\n--- DEBUG: Inspecting all_pois_gdf before cleaning ---')
print(f"Shape of all_pois_gdf: {all_pois_gdf.shape}")
print(f"Null counts in 'osm_id': {all_pois_gdf['osm_id'].isnull().sum()}")
print(f"Null counts in 'shape_id': {all_pois_gdf['shape_id'].isnull().sum()}")
print(f"Unique geometry types before filtering: {all_pois_gdf.geometry.geom_type.unique()}")
print("Head of all_pois_gdf before cleaning:")
display(all_pois_gdf.head())


--- DEBUG: Inspecting all_pois_gdf before cleaning ---
Shape of all_pois_gdf: (91121, 10)
Null counts in 'osm_id': 0
Null counts in 'shape_id': 0
Unique geometry types before filtering: ['Point' 'Polygon' 'MultiPolygon']
Head of all_pois_gdf before cleaning:


,osm_id,geometry,element_type,name,amenity,shop,office,building,healthcare,shape_id
0,MISSING_OSMID_10106110_0_5ee2a577,POINT (36.81492 -1.27217),NaN,Aga Khan Nursery,school,NaN,NaN,NaN,NaN,10106110
1,MISSING_OSMID_10106110_1_7423d94a,POINT (36.80451 -1.23001),NaN,City Walk - Village Market,NaN,shoes,NaN,NaN,NaN,10106110
2,MISSING_OSMID_10106110_2_bad06261,POINT (36.80146 -1.22415),NaN,River Garden Centre,NaN,garden_centre,NaN,NaN,NaN,10106110
3,MISSING_OSMID_10106110_3_ba58e30a,POINT (36.82025 -1.24999),NaN,Onn The Way Supermarket Muthaiga,NaN,supermarket,NaN,NaN,NaN,10106110
4,MISSING_OSMID_10106110_4_03c59614,POINT (36.80496 -1.22869),NaN,Dorman,NaN,mall,NaN,NaN,NaN,10106110


#### 6. Clean `all_pois_gdf`

Based on the validation warnings, we will clean `all_pois_gdf` by:
- Filtering to include only `Point`, `Polygon`, or `MultiPolygon` geometries.
- Dropping rows where `osm_id` or `shape_id` are null, as these are essential identifiers.

In [52]:
print('\n--- Cleaning all_pois_gdf ---')

initial_pois_count = len(all_pois_gdf)
print(f"Initial POI count: {initial_pois_count}")

# Filter out invalid geometries (keep only Point, Polygon, MultiPolygon)
valid_geometry_types = ['Point', 'Polygon', 'MultiPolygon']
all_pois_gdf = all_pois_gdf[all_pois_gdf.geometry.geom_type.isin(valid_geometry_types)]
print(f"POI count after filtering for valid geometries: {len(all_pois_gdf)}")

# Drop rows with null 'osm_id' or 'shape_id'
all_pois_gdf.dropna(subset=['osm_id', 'shape_id'], inplace=True)
print(f"POI count after dropping nulls in 'osm_id' or 'shape_id': {len(all_pois_gdf)}")

# Convert 'osm_id' to integer type if it contains float values due to NaNs
# It's common for OSM IDs to be large integers, so check if conversion is safe.
# First ensure there are no NaNs left, then convert.
if not all_pois_gdf['osm_id'].isnull().any():
    try:
        all_pois_gdf['osm_id'] = all_pois_gdf['osm_id'].astype(int)
        print("Converted 'osm_id' to integer type.")
    except ValueError:
        print("Warning: Could not convert 'osm_id' to integer type, keeping original type.")

print('all_pois_gdf cleaning complete.')
display(all_pois_gdf.head())


--- Cleaning all_pois_gdf ---
Initial POI count: 91123
POI count after filtering for valid geometries: 91121
POI count after dropping nulls in 'osm_id' or 'shape_id': 91121
all_pois_gdf cleaning complete.


/tmp/ipykernel_1210/750775150.py:12: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  all_pois_gdf.dropna(subset=['osm_id', 'shape_id'], inplace=True)


,osm_id,geometry,element_type,name,amenity,shop,office,building,healthcare,shape_id
0,MISSING_OSMID_10106110_0_5ee2a577,POINT (36.81492 -1.27217),NaN,Aga Khan Nursery,school,NaN,NaN,NaN,NaN,10106110
1,MISSING_OSMID_10106110_1_7423d94a,POINT (36.80451 -1.23001),NaN,City Walk - Village Market,NaN,shoes,NaN,NaN,NaN,10106110
2,MISSING_OSMID_10106110_2_bad06261,POINT (36.80146 -1.22415),NaN,River Garden Centre,NaN,garden_centre,NaN,NaN,NaN,10106110
3,MISSING_OSMID_10106110_3_ba58e30a,POINT (36.82025 -1.24999),NaN,Onn The Way Supermarket Muthaiga,NaN,supermarket,NaN,NaN,NaN,10106110
4,MISSING_OSMID_10106110_4_03c59614,POINT (36.80496 -1.22869),NaN,Dorman,NaN,mall,NaN,NaN,NaN,10106110


#### 6. Clean `all_pois_gdf`

Based on the validation warnings, we will clean `all_pois_gdf` by:
- Filtering to include only `Point`, `Polygon`, or `MultiPolygon` geometries.
- Dropping rows where `osm_id` or `shape_id` are null, as these are essential identifiers.

In [56]:
print('\n--- Cleaning all_pois_gdf ---')

initial_pois_count = len(all_pois_gdf)
print(f"Initial POI count: {initial_pois_count}")

# Filter out invalid geometries (keep only Point, Polygon, MultiPolygon)
valid_geometry_types = ['Point', 'Polygon', 'MultiPolygon']
# Make a copy to avoid SettingWithCopyWarning later and ensure independence
cleaned_pois_gdf = all_pois_gdf[all_pois_gdf.geometry.geom_type.isin(valid_geometry_types)].copy()
print(f"POI count after filtering for valid geometries: {len(cleaned_pois_gdf)}")

# Explicitly check nulls right before dropping them to understand the discrepancy
print(f"Null counts in 'osm_id' before dropna: {cleaned_pois_gdf['osm_id'].isnull().sum()}")
print(f"Null counts in 'shape_id' before dropna: {cleaned_pois_gdf['shape_id'].isnull().sum()}")

# Drop rows with null 'osm_id' or 'shape_id'
cleaned_pois_gdf.dropna(subset=['osm_id', 'shape_id'], inplace=True)
print(f"POI count after dropping nulls in 'osm_id' or 'shape_id': {len(cleaned_pois_gdf)}")

# Convert 'osm_id' to integer type if it contains float values due to NaNs
# It's common for OSM IDs to be large integers, so check if conversion is safe.
# First ensure there are no NaNs left, then convert.
if not cleaned_pois_gdf['osm_id'].isnull().any():
    try:
        # Check if any 'MISSING_OSMID_' strings are present. If so, conversion to int will fail.
        if cleaned_pois_gdf['osm_id'].astype(str).str.contains('MISSING_OSMID_').any():
            print("Warning: 'osm_id' contains placeholder strings, cannot convert to integer. Keeping original type.")
        else:
            cleaned_pois_gdf['osm_id'] = cleaned_pois_gdf['osm_id'].astype(int)
            print("Converted 'osm_id' to integer type.")
    except ValueError:
        print("Warning: Could not convert 'osm_id' to integer type (e.g., non-numeric values), keeping original type.")

print('all_pois_gdf cleaning complete.')
display(cleaned_pois_gdf.head())

# Reassign back to all_pois_gdf for subsequent steps
all_pois_gdf = cleaned_pois_gdf


--- Cleaning all_pois_gdf ---
Initial POI count: 91123
POI count after filtering for valid geometries: 91121
Null counts in 'osm_id' before dropna: 0
Null counts in 'shape_id' before dropna: 0
POI count after dropping nulls in 'osm_id' or 'shape_id': 91121
all_pois_gdf cleaning complete.


,osm_id,geometry,element_type,name,amenity,shop,office,building,healthcare,shape_id
0,MISSING_OSMID_10106110_0_5ee2a577,POINT (36.81492 -1.27217),None,Aga Khan Nursery,school,None,None,None,None,10106110
1,MISSING_OSMID_10106110_1_7423d94a,POINT (36.80451 -1.23001),None,City Walk - Village Market,None,shoes,None,None,None,10106110
2,MISSING_OSMID_10106110_2_bad06261,POINT (36.80146 -1.22415),None,River Garden Centre,None,garden_centre,None,None,None,10106110
3,MISSING_OSMID_10106110_3_ba58e30a,POINT (36.82025 -1.24999),None,Onn The Way Supermarket Muthaiga,None,supermarket,None,None,None,10106110
4,MISSING_OSMID_10106110_4_03c59614,POINT (36.80496 -1.22869),None,Dorman,None,mall,None,None,None,10106110


### Data Summary for each source

In [44]:
print('--- EPRA Fuel Prices (epra_df) Summary ---')
print(f'Shape: {epra_df.shape}')
print('\nData Types:')
print(epra_df.dtypes)
print(f"\nDate Range: {epra_df['From_Date'].min().strftime('%Y-%m-%d')} to {epra_df['To_Date'].max().strftime('%Y-%m-%d')}")
print('\nNull Counts:')
print(epra_df.isnull().sum())

--- EPRA Fuel Prices (epra_df) Summary ---
Shape: (19, 7)

Data Types:
From                    object
To                      object
Town                    object
Super (PMS)            float64
Diesel (AGO)           float64
From_Date       datetime64[ns]
To_Date         datetime64[ns]
dtype: object

Date Range: 2024-11-15 to 2026-05-14

Null Counts:
From            0
To              0
Town            0
Super (PMS)     0
Diesel (AGO)    0
From_Date       0
To_Date         0
dtype: int64


In [45]:
print('\n--- Hourly Weather Data (hourly_df) Summary ---')
print(f'Shape: {hourly_df.shape}')
print('\nData Types:')
print(hourly_df.dtypes)
print(f"\nDate Range: {hourly_df['time'].min().strftime('%Y-%m-%d %H:%M')} to {hourly_df['time'].max().strftime('%Y-%m-%d %H:%M')}")
print('\nNull Counts:')
print(hourly_df.isnull().sum())


--- Hourly Weather Data (hourly_df) Summary ---
Shape: (18984, 4)

Data Types:
time             datetime64[ns]
temperature_c           float64
rain                    float64
wind_speed              float64
dtype: object

Date Range: 2024-01-01 00:00 to 2026-03-01 23:00

Null Counts:
time             0
temperature_c    0
rain             0
wind_speed       0
dtype: int64


In [46]:
print('\n--- GTFS Routes Data (routes_df) Summary ---')
print(f'Shape: {routes_df.shape}')
print('\nData Types:')
print(routes_df.dtypes)
print('\nNull Counts:')
print(routes_df.isnull().sum())


--- GTFS Routes Data (routes_df) Summary ---
Shape: (136, 5)

Data Types:
route_id            string[python]
agency_id           string[python]
route_short_name    string[python]
route_long_name     string[python]
route_type                   Int32
dtype: object

Null Counts:
route_id            0
agency_id           0
route_short_name    0
route_long_name     0
route_type          0
dtype: int64


In [47]:
print('\n--- GTFS Stops Data (stops_df) Summary ---')
print(f'Shape: {stops_df.shape}')
print('\nData Types:')
print(stops_df.dtypes)
print('\nNull Counts:')
print(stops_df.isnull().sum())


--- GTFS Stops Data (stops_df) Summary ---
Shape: (4273, 6)

Data Types:
stop_id           string[python]
stop_name         string[python]
stop_lat                 float64
stop_lon                 float64
location_type              Int32
parent_station    string[python]
dtype: object

Null Counts:
stop_id              0
stop_name            0
stop_lat             0
stop_lon             0
location_type     4221
parent_station    4273
dtype: int64


In [48]:
print('\n--- GTFS Stop Times Data (stop_times_df) Summary ---')
print(f'Shape: {stop_times_df.shape}')
print('\nData Types:')
print(stop_times_df.dtypes)
print('\nNull Counts:')
print(stop_times_df.isnull().sum())


--- GTFS Stop Times Data (stop_times_df) Summary ---
Shape: (7533, 5)

Data Types:
trip_id           string[python]
arrival_time      string[python]
departure_time    string[python]
stop_id           string[python]
stop_sequence              Int32
dtype: object

Null Counts:
trip_id           0
arrival_time      0
departure_time    0
stop_id           0
stop_sequence     0
dtype: int64


In [49]:
print('\n--- GTFS Shapes Data (shapes_from_feed_df) Summary ---')
print(f'Shape: {shapes_from_feed_df.shape}')
print('\nData Types:')
print(shapes_from_feed_df.dtypes)
print('\nNull Counts:')
print(shapes_from_feed_df.isnull().sum())


--- GTFS Shapes Data (shapes_from_feed_df) Summary ---
Shape: (36483, 4)

Data Types:
shape_id             string[python]
shape_pt_lat                float64
shape_pt_lon                float64
shape_pt_sequence             Int32
dtype: object

Null Counts:
shape_id             0
shape_pt_lat         0
shape_pt_lon         0
shape_pt_sequence    0
dtype: int64


In [50]:
print('\n--- Route Geometries (routes_gdf) Summary ---')
print(f'Shape: {routes_gdf.shape}')
print('\nData Types:')
print(routes_gdf.dtypes)
print('\nNull Counts:')
print(routes_gdf.isnull().sum())


--- Route Geometries (routes_gdf) Summary ---
Shape: (272, 2)

Data Types:
shape_id      object
geometry    geometry
dtype: object

Null Counts:
shape_id    0
geometry    0
dtype: int64


In [51]:
print('\n--- Points of Interest (all_pois_gdf) Summary ---')
print(f'Shape: {all_pois_gdf.shape}')
print('\nData Types:')
print(all_pois_gdf.dtypes)
print('\nNull Counts:')
print(all_pois_gdf.isnull().sum())


--- Points of Interest (all_pois_gdf) Summary ---
Shape: (91123, 10)

Data Types:
osm_id            object
geometry        geometry
element_type      object
name              object
amenity           object
shop              object
office            object
building          object
healthcare        object
shape_id          object
dtype: object

Null Counts:
osm_id              0
geometry            0
element_type    91123
name             9650
amenity         58385
shop            61072
office          79503
building        63827
healthcare      82382
shape_id            0
dtype: int64


In [55]:
print('\n--- DEBUG: Inspecting all_pois_gdf before cleaning ---')
print(f"Shape of all_pois_gdf: {all_pois_gdf.shape}")
print(f"Null counts in 'osm_id': {all_pois_gdf['osm_id'].isnull().sum()}")
print(f"Null counts in 'shape_id': {all_pois_gdf['shape_id'].isnull().sum()}")
print(f"Unique geometry types before filtering: {all_pois_gdf.geometry.geom_type.unique()}")
print("Head of all_pois_gdf before cleaning:")
display(all_pois_gdf.head())


--- DEBUG: Inspecting all_pois_gdf before cleaning ---
Shape of all_pois_gdf: (91123, 10)
Null counts in 'osm_id': 0
Null counts in 'shape_id': 0
Unique geometry types before filtering: ['Point' 'Polygon' 'MultiPolygon' 'LineString']
Head of all_pois_gdf before cleaning:


,osm_id,geometry,element_type,name,amenity,shop,office,building,healthcare,shape_id
0,MISSING_OSMID_10106110_0_5ee2a577,POINT (36.81492 -1.27217),None,Aga Khan Nursery,school,None,None,None,None,10106110
1,MISSING_OSMID_10106110_1_7423d94a,POINT (36.80451 -1.23001),None,City Walk - Village Market,None,shoes,None,None,None,10106110
2,MISSING_OSMID_10106110_2_bad06261,POINT (36.80146 -1.22415),None,River Garden Centre,None,garden_centre,None,None,None,10106110
3,MISSING_OSMID_10106110_3_ba58e30a,POINT (36.82025 -1.24999),None,Onn The Way Supermarket Muthaiga,None,supermarket,None,None,None,10106110
4,MISSING_OSMID_10106110_4_03c59614,POINT (36.80496 -1.22869),None,Dorman,None,mall,None,None,None,10106110
